# LunaQuest / Lunar IceNav Research Notebook

Research notebook for BAH 2026 Problem Statement 8. Outputs are radar-based candidate screening products, scientific review tables, preliminary landing candidates, conceptual rover routes, pseudo-label ML experiments, and validation reports.

## A. Research Objective and C8 Problem

Detect radar-based candidate subsurface ice regions in the lunar south polar Faustini/F2 AOI, characterize candidate patch extent and uncertainty, and connect candidates to landing and rover planning constraints.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'pipeline.json').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not find project root containing configs/pipeline.json and src/')

root = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(root / 'src'))
figdir = root / 'outputs' / 'figures'
tables = root / 'outputs' / 'tables'
routes_dir = root / 'outputs' / 'routes'
print(root)

## B. Research Papers and Method Traceability

Maps each uploaded method reference to the module it supports, with duplicate copies ignored.

In [ ]:
import pandas as pd
from IPython.display import display, Image
import json
trace = pd.read_csv(tables / 'research_method_traceability.csv')
display(trace)
print((root / 'reports' / 'RESEARCH_PAPER_METHOD_MAP.md').read_text()[:4000])

## C. Region Validation: Faustini/F2 AOI

Configured AOI: lat -87.8 to -86.9, lon 80 to 85 E. Product selection is based on actual AOI coverage.

In [ ]:
manifest = json.loads((root / 'reports' / 'run_manifest.json').read_text())
print('AOI:', manifest['aoi'])
display(pd.read_csv(tables / 'sar_product_selection_reason.csv')[['product_id','coverage_fraction','selected_for_main_map','selection_reason']])
display(Image(filename=str(figdir / 'region_validation_map.png')))

## D. Dataset Inventory and Product Selection

Inventory, coverage, and data-status tables record what was used and what remains context-only or future-required.

In [ ]:
inv = pd.read_csv(tables / 'product_inventory.csv')
display(inv['product_type'].value_counts().reset_index(name='rows'))
display(pd.read_csv(tables / 'data_coverage_status.csv'))
display(Image(filename=str(figdir / '01_dataset_inventory_chart.png')))
display(Image(filename=str(figdir / '02_selected_sar_coverage.png')))

## E. SAR Feature Extraction

Features include SAR log intensity, LH/LV and LV/LH ratio proxies, polarization imbalance proxy, local texture, local mean/std, and candidate score. Calibrated CPR/DOP require future complex/Stokes data.

In [ ]:
sar_meta = pd.read_csv(tables / 'selected_sar_metadata.csv')
display(sar_meta.T)
features = __import__('numpy').load(root / 'outputs' / 'features' / 'sar_feature_stack.npz')
print('Feature arrays:', list(features.keys()))
display(Image(filename=str(figdir / 'radar_feature_panel.png')))

## F. Radar-Based Candidate Ice Map

The candidate map is generated before landing analysis from SAR screening proxies and connected components.

In [ ]:
candidate_summary = pd.read_csv(tables / 'candidate_patch_summary.csv')
display(candidate_summary)
display(Image(filename=str(figdir / 'ice_candidate_detection_map.png')))
display(Image(filename=str(figdir / 'radar_candidate_overlay.png')))

## G. Candidate Patch Evaluation

Each patch is evaluated for area, equivalent candidate patch diameter, score, ratio proxy, texture, slope, confidence, landing proximity, and route accessibility.

In [ ]:
review = pd.read_csv(tables / 'candidate_scientific_review.csv')
display(review.head(15))
for name in ['ice_candidate_confidence_map.png','candidate_area_vs_score_scatter.png','candidate_diameter_distribution.png','top_candidate_review_panel.png']:
    display(Image(filename=str(figdir / name)))

## G2. Science Justification Layer

Judge-facing explanations connect the selected patch, landing site, and rover route to the actual screening metrics.

In [ ]:
patch_table = pd.read_csv(tables / 'candidate_patch_table.csv')
display(patch_table.head(10))
print((root / 'reports' / 'justification_report.txt').read_text())
for name in ['science_justification_overlay.png','ice_probability_map_annotated.png','landing_site_map_annotated.png','rover_routes_annotated.png']:
    display(Image(filename=str(figdir / name)))

## H. Radar Ambiguity and Roughness Penalty

High radar response can be caused by rough terrain or multiple scattering, so candidates are retained but flagged with roughness ambiguity risk.

In [ ]:
patch_review = pd.read_csv(tables / 'ice_candidate_patch_review.csv')
display(patch_review.head(15))
for name in ['radar_roughness_ambiguity_map.png','candidate_score_vs_roughness.png','candidate_score_vs_slope.png']:
    display(Image(filename=str(figdir / name)))

## I. Threshold Sensitivity and Confidence

Candidate stability is evaluated across score thresholds from 0.60 to 0.85 and contributes to confidence.

In [ ]:
thresh = pd.read_csv(tables / 'threshold_sensitivity.csv')
display(thresh)
for name in ['threshold_sensitivity_curve.png','candidate_stability_map.png']:
    display(Image(filename=str(figdir / name)))

## I2. PSR Proxy and Depth Likelihood

The PSR layer here is an approximation from poleward latitude plus terrain-shadow context. Depth likelihood is a rule-based planning label from radar score and roughness.

In [ ]:
for name in ['psr_stability_proxy_map.png','ice_confidence_score_map.png','depth_likelihood_map.png']:
    display(Image(filename=str(figdir / name)))

## J. Scenario-Based Resource Estimate

Resource scenarios are planning-only estimates for later-validated candidate patches.

In [ ]:
resource = pd.read_csv(tables / 'resource_scenario_estimates.csv')
display(resource.head(24))
display(Image(filename=str(figdir / 'resource_scenario_bar_chart.png')))

## K. DEM/DTM Slope and Roughness

DEM-derived slope, roughness, and hillshade support terrain safety and route screening.

In [ ]:
slope_summary = pd.read_csv(tables / 'slope_safety_summary.csv')
display(slope_summary)
for name in ['dem_elevation_map.png','dtm_slope_map.png','slope_classification_map.png','terrain_roughness_map.png','hillshade_map.png']:
    display(Image(filename=str(figdir / name)))

## L. Fuzzy Landing Site Scoring

The landing model uses candidate proximity, slope safety, roughness avoidance, and neutral placeholders for missing illumination/temperature layers.

In [ ]:
landing = pd.read_csv(tables / 'fuzzy_landing_site_scores.csv')
display(landing)
for name in ['fuzzy_landing_score_map.png','landing_score_components.png','landing_candidate_decision_map.png']:
    display(Image(filename=str(figdir / name)))

## M. Landing Site vs Crater/Candidate-Mask Validation

Landing candidates are checked against configured AOI, candidate mask, unsafe slope, and F2 boundary availability.

In [ ]:
boundary = pd.read_csv(tables / 'landing_crater_boundary_check.csv')
display(boundary)
for name in ['landing_vs_f2_crater_boundary_map.png','landing_to_candidate_distance_map.png']:
    display(Image(filename=str(figdir / name)))

## N. Rover Navigation and Route Evaluation

A* route variants compare shortest, safest, science-priority, and energy-aware planning concepts using available slope/roughness/science proxy layers.

In [ ]:
rover = pd.read_csv(tables / 'rover_navigation_evaluation.csv')
display(rover[['route_type','route_recommendation','route_decision_score','target_candidate_id','start_landing_site_id','length_m','total_cost','percent_under_5deg','science_reward_score','energy_cost_proxy','traverse_risk_score']])
for name in ['rover_route_decision_map.png','rover_route_slope_profile.png','rover_route_risk_profile.png','rover_route_comparison_chart.png']:
    display(Image(filename=str(figdir / name)))

## N2. Rover Traversal Simulation

The recommended route is sampled step-by-step with cumulative energy proxy and slope-vs-distance profiles.

In [ ]:
traversal = pd.read_csv(tables / 'rover_traversal_simulation.csv')
display(traversal.head())
display(traversal.tail())
for name in ['rover_energy_profile.png','rover_slope_vs_distance.png','rover_traversal_steps.png']:
    display(Image(filename=str(figdir / name)))
print('Animation:', figdir / 'rover_traversal_animation.gif')

## O. U-Net Pseudo-Label Experiment

The U-Net is weakly supervised with rule-based pseudo-labels. Metrics are pseudo-label agreement metrics.

In [ ]:
metrics = pd.read_csv(tables / 'unet_pseudo_label_metrics.csv')
experiments = pd.read_csv(tables / 'model_experiment_comparison.csv')
display(metrics.T)
display(experiments)
for name in ['unet_training_curve.png','model_experiment_comparison.png','unet_prediction_overlay.png','unet_error_map_against_pseudolabel.png']:
    display(Image(filename=str(figdir / name)))

## P. External Validation Layers Status

Diviner, PSR/illumination, LOLA albedo, LAMP, and M3 layers are not fabricated; missing data are listed as future-required.

In [ ]:
validation = pd.read_csv(tables / 'validation_layer_status.csv')
candidate_validation = pd.read_csv(tables / 'candidate_validation_against_external_layers.csv')
display(validation)
display(candidate_validation.head(15))
display(Image(filename=str(figdir / 'validation_layer_availability_matrix.png')))

## Q. Final Research Evaluation Summary

The evaluation report links data, methods, candidate patches, landing scoring, route selection, ML agreement, limitations, and validation needs.

In [ ]:
print((root / 'reports' / 'MODEL_EVALUATION_REPORT.md').read_text()[:7000])
display(Image(filename=str(figdir / 'combined_research_decision_map.png')))

## R. Technical Limitations and Next Data

Limitations and next-download priorities are explicit so the prototype does not overclaim.

In [ ]:
print((root / 'reports' / 'TECHNICAL_LIMITATIONS.md').read_text())
print('\n' + '='*80 + '\n')
print((root / 'reports' / 'NEXT_DATA_TO_DOWNLOAD.md').read_text())